# Are the MI pruning assumptions met?

[mi_importance.py](mi_importance.py) scores a channel with a closed-form Gaussian conditional MI.
That formula is exact only if the variables going into it are **jointly Gaussian**. This notebook
checks whether they are.

Part 1 (below) pins down which variables those are and looks at their marginals.

## Setup

In [ ]:
!git clone --branch mipp-lookahead2 https://github.com/elliotcanter11/Diff-Pruning.git

In [ ]:
%cd Diff-Pruning/

In [ ]:
!pip install -r requirements.txt

In [ ]:
!python tools/extract_cifar10_hug.py --output data

In [ ]:
!bash tools/convert_cifar10_ddpm_ema.sh

## Capture

The same calibration loop `ddpm_prune.py` runs before pruning, so the buffers below hold exactly
what the estimator sees.

In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.auto import tqdm

# compat shims, copied from ddpm_prune.py (diffusers/ here is the vendored copy)
import huggingface_hub
from huggingface_hub import constants as hf_constants
if not hasattr(hf_constants, "hf_cache_home"):
    hf_constants.hf_cache_home = hf_constants.HF_HUB_CACHE
if not hasattr(huggingface_hub, "cached_download"):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download
if not hasattr(huggingface_hub, "HfFolder"):
    class HfFolder:
        @staticmethod
        def get_token(): return huggingface_hub.get_token()
    huggingface_hub.HfFolder = HfFolder
import jax
if not hasattr(jax.random, "KeyArray"): jax.random.KeyArray = jax.Array
import transformers.utils as tf_utils
if not hasattr(tf_utils, "FLAX_WEIGHTS_NAME"): tf_utils.FLAX_WEIGHTS_NAME = "flax_model.msgpack"

from diffusers import DDPMPipeline
from mi_importance import MIImportance
from torchvision import transforms as T
import utils

DEVICE  = 'cuda:0'
BATCH   = 128
BATCHES = 16     # BATCH*BATCHES images
LOCS    = 4      # = --mi_num_locations

pipeline = DDPMPipeline.from_pretrained('pretrained/ddpm_ema_cifar10').to(DEVICE)
model, scheduler = pipeline.unet.eval(), pipeline.scheduler

tf = T.Compose([T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean=0.5, std=0.5)])
loader = torch.utils.data.DataLoader(
    utils.get_dataset('data/cifar10_images', transform=tf),
    batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)

imp = MIImportance(num_locations=LOCS).attach(model, ignored_layers=[model.conv_out])

t_raw, it = [], iter(loader)
with torch.no_grad():
    for _ in tqdm(range(BATCHES), desc='calibrating'):
        batch = next(it)
        batch = batch[0] if isinstance(batch, (list, tuple)) else batch
        batch = batch.to(DEVICE)
        t = torch.randint(0, scheduler.config.num_train_timesteps, (batch.shape[0],), device=DEVICE).long()
        noisy = scheduler.add_noise(batch, torch.randn_like(batch), t)
        imp.new_pass(batch.shape[0])
        out = model(noisy, t).sample
        imp.record_output(out)
        imp.record_timesteps(t)
        t_raw.append(t.cpu())
imp.finalize()

convs = [m for m, _ in sorted(imp._order.items(), key=lambda kv: kv[1]) if imp._loc_buf.get(m) is not None]
name_of = {m: n for n, m in model.named_modules()}

t_img  = torch.cat(t_raw).numpy()                 # (n_images,)   timestep per image
t_loc  = np.repeat(t_img, LOCS)                   # (n_rows,)     timestep per (image, pixel)
coords = torch.cat(imp._coords_buf).numpy()       # (n_rows, 2)   normalised (u, v) of each pixel

loc = lambda m: imp._loc_buf[m].float().numpy()   # (n_rows, C)   channels at sampled pixels
img = lambda m: imp._img_buf[m].float().numpy()   # (n_images, C) channels pooled over space
out_target = imp._out_target.numpy()              # (n_images, 192) predicted noise, pooled to 8x8x3

COND     = imp._loc_cond.numpy()                  # (n_rows, 15)   [t features | position features]
COND_IMG = imp._img_cond.numpy()                  # (n_images, 9)  [t features]

print(f'{len(convs)} convs | {len(t_img)} images | {len(t_loc)} (image, pixel) rows')

## Which variables are assumed jointly Gaussian?

Each term stacks three blocks into one vector `Z = [X | cond | T]`, takes its covariance, and reads
MI off the inverse. So there is **one joint-Gaussianity assumption per term**, over a different
vector, with a different definition of "a sample".

**Adjacency term** — a sample is one **(image, pixel)**, giving `n_images × num_locations` rows.

| block | what it is | width |
|---|---|---|
| `X` | the root conv's channels at that pixel | `C` |
| `T` | the consumer convs' channels at the *same* pixel | ≤ `target_dim_cap` (512) |
| `cond` | 9 Fourier features of `t`, 6 smooth features of position `(u,v)` | 15 |

**Mid-range term** — identical, except `T` is a single conv about halfway to the output.

**Output term** — a sample is one **image**, giving `n_images` rows.

| block | what it is | width |
|---|---|---|
| `X` | each root channel pooled to `g×g` (default `g=1`, so the channel's spatial mean) | `C·g²` |
| `T` | the predicted noise, pooled to 8×8×3 | 192 |
| `cond` | 9 Fourier features of `t` | 9 |

### What the assumption actually says

`cond` is a deterministic basis of `(t, u, v)` — bounded and functionally dependent
(`sin² + cos² = 1`), so it is *not* Gaussian and never could be. Putting it in the covariance only
means the estimator linearly projects it out. So the real assumption is about the activations:

> Given `t` and position, `[X, T]` is jointly Gaussian, with a mean linear in the `cond` basis and a
> covariance that does not depend on `t` or position.

which is three separate claims, each testable:

1. every coordinate of `X` and `T` is Gaussian on its own — **necessary, not sufficient**
2. `X` and `T` are *jointly* Gaussian, not just Gaussian one at a time
3. the covariance is the same at every `t` and every position

Part 1 below is claim 1.

In [ ]:
# The four blocks whose marginals the estimator relies on, for one layer.
LAYER = convs[len(convs)//2]                      # change this to move around the network
TARGET = convs[imp._order[LAYER] + 1]             # stand-in for the group's consumer convs

FAMILIES = {
    'adjacency X': (loc(LAYER),   'root channels at a pixel'),
    'adjacency T': (loc(TARGET),  'next conv channels at the same pixel'),
    'output X':    (img(LAYER),   'root channels, pooled over space'),
    'output T':    (out_target,   'predicted noise, pooled over space'),
}

print(f'LAYER  = [{imp._order[LAYER]}] {name_of[LAYER]}')
print(f'TARGET = [{imp._order[TARGET]}] {name_of[TARGET]}\n')
for k, (A, desc) in FAMILIES.items():
    print(f'  {k:<13} {A.shape[0]:>6} rows x {A.shape[1]:>4} cols   {desc}')

## Part 1 — Marginals

Joint Gaussianity implies every coordinate is Gaussian on its own, so a marginal failure rules the
assumption out immediately. Passing does **not** rule it in; that is Part 2.

Both figures pool all timesteps and positions together, which is what the estimator's covariance
sees.

In [ ]:
plt.rcParams.update({'figure.dpi': 120, 'font.size': 8.5, 'axes.titlesize': 8.5,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'axes.axisbelow': True})
BLUE, RED, GREY = '#3b6fb6', '#d1495b', '#8d99ae'

def qq(ax, x, n_pts=700):
    """QQ plot of x against a standard normal, with the identity line as reference."""
    x = np.sort((x - x.mean()) / (x.std() + 1e-12))
    i = np.unique(np.linspace(0, len(x) - 1, min(n_pts, len(x))).astype(int))
    ax.axline((0, 0), slope=1, color=RED, lw=1.1, zorder=1)
    ax.plot(stats.norm.ppf((i + 0.5) / len(x)), x[i], 'o', ms=2, color=BLUE, alpha=.75, zorder=2)
    ax.set_xlim(-4.2, 4.2)

def col_moments(A, max_cols=400, seed=0):
    """Skew and excess kurtosis of every column (subsampled if the block is very wide)."""
    cols = np.sort(np.random.default_rng(seed).choice(
        A.shape[1], min(max_cols, A.shape[1]), replace=False))
    Z = A[:, cols].astype(np.float64)
    return cols, stats.skew(Z, 0), stats.kurtosis(Z, 0)

STATS = {fam: col_moments(A) for fam, (A, _) in FAMILIES.items()}

In [ ]:
# Figure 1 -- QQ plots for coordinates at four kurtosis percentiles of each block, so the
# panels deliberately span each block's range instead of landing wherever chance puts them.
QUANTILES = [(0.10, 'p10'), (0.50, 'p50'), (0.90, 'p90'), (0.99, 'p99')]
fig, axes = plt.subplots(len(FAMILIES), len(QUANTILES),
                         figsize=(2.25*len(QUANTILES), 2.05*len(FAMILIES)), sharex=True)
for r, (fam, (A, _)) in enumerate(FAMILIES.items()):
    cols, _, kurt = STATS[fam]
    order = np.argsort(kurt)
    for c, (q, tag) in enumerate(QUANTILES):
        k = order[int(round(q * (len(order) - 1)))]
        qq(axes[r, c], A[:, cols[k]].astype(np.float64))
        axes[r, c].set_title(f'{tag}   col {cols[k]}   kurt {kurt[k]:+.1f}', pad=3)
    axes[r, 0].set_ylabel(f'{fam}\nobserved', fontsize=8)
for c in range(len(QUANTILES)):
    axes[-1, c].set_xlabel('normal quantile')
fig.suptitle('Marginal QQ plots — on the red line = Gaussian', y=1.005, fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# Figure 2 -- skew and excess kurtosis of EVERY coordinate, to see whether figure 1 is typical
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
names = list(FAMILIES)
rng = np.random.default_rng(2)

for ax, k, label in zip(axes, (1, 2), ('skew', 'excess kurtosis')):
    for i, fam in enumerate(names):
        d = STATS[fam][k]
        y = len(names) - 1 - i
        ax.plot(d, y + rng.uniform(-.17, .17, len(d)), 'o', ms=2, color=GREY, alpha=.45, zorder=2)
        q1, med, q3 = np.percentile(d, [25, 50, 75])
        ax.plot([q1, q3], [y, y], color=BLUE, lw=5, solid_capstyle='round', alpha=.9, zorder=3)
        ax.plot([med], [y], marker='|', ms=11, mew=1.8, color='white', zorder=4)
    ax.axvline(0, color=RED, lw=1.1, zorder=1)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names[::-1])
    ax.set_ylim(-.6, len(names) - .4)
    # both span orders of magnitude, so a few extreme channels would flatten a linear axis
    ax.set_xscale('symlog', linthresh=1)
    ax.set_xlabel(f'{label}   (0 = Gaussian, symlog scale)')
    ax.grid(axis='y', visible=False)
fig.suptitle('Every coordinate — bar = interquartile range, tick = median', y=1.03, fontsize=10)
fig.tight_layout()
plt.show()

### Reading these

**Figure 1** — points on the red line means Gaussian. Both ends bending away from the line means
tails heavier than Gaussian; one end only means skew. The four columns are the coordinates at the
10th, 50th, 90th and 100th percentile of kurtosis within that block, so the row shows you its range.

**Figure 2** — where all the coordinates sit. Two things matter:

* how far from 0 the bulk sits — heavy tails mean a few extreme rows set the covariance, and the
  covariance is the estimator's only input;
* how *spread out* each row is — non-Gaussianity that hits every channel equally largely cancels
  out of a ranking, but channels that differ from each other get ranked partly on tail shape.

Expect `output X` and `output T` to look tamer than the adjacency blocks: they are averages over
H×W, so the CLT works in their favour.

One caveat before concluding anything from heavy tails here: both figures pool every timestep
together, and **a mixture of Gaussians with different scales is heavy-tailed even when each slice is
perfectly Gaussian**. Claim 3 is what separates those two explanations.

## Part 2 — Joint Gaussianity

Every marginal can pass while the joint fails, and the joint is what the determinant formula uses.

For a block of columns, the squared **Mahalanobis distance**

    d²ᵢ = (xᵢ − x̄)ᵀ S⁻¹ (xᵢ − x̄)

measures how far row `i` sits from the centre in units of the fitted covariance. If the block is
jointly Gaussian these follow a known distribution, so sorting them and plotting against that
distribution's quantiles should give a straight line — a QQ plot for the whole vector at once.

Three things that make the figure trustworthy:

* `S` is estimated from the same rows, so the reference is **not** `χ²_d`. The exact law is a scaled
  Beta, and that is what the red line uses.
* Each block is residualized on `cond` first, since that is exactly what the estimator does with it.
  So this tests the joint Gaussianity of what the covariance actually sees.
* The last panel is **synthetic Gaussian data of the same shape** — a control showing what passing
  looks like at this `n` and `d`.

We test random sub-blocks, because `n` has to comfortably exceed `d`. That is sound in one
direction: joint Gaussianity of the whole vector implies it for every sub-block, so **a sub-block
that fails falsifies the assumption**. A sub-block that passes is suggestive, not proof.

In [ ]:
D = 32          # columns per block; keep well below the row count

def pick(A, k, seed):
    cols = np.sort(np.random.default_rng(seed).choice(A.shape[1], min(k, A.shape[1]), replace=False))
    return A[:, cols].astype(np.float64)

_n_loc = loc(LAYER).shape[0]
BLOCKS = {
    'adjacency X':   (pick(loc(LAYER), D, 0), COND),
    'adjacency T':   (pick(loc(TARGET), D, 1), COND),
    'adjacency X+T': (np.hstack([pick(loc(LAYER), D//2, 2), pick(loc(TARGET), D//2, 3)]), COND),
    'output X+T':    (np.hstack([pick(img(LAYER), D//2, 4), pick(out_target, D//2, 5)]), COND_IMG),
    'gaussian control': (np.random.default_rng(6).standard_normal((_n_loc, D)), COND),
}

def mahalanobis_qq(X, cond):
    """Squared Mahalanobis distances of X (after projecting out cond), and the quantiles they
    would follow if X were jointly Gaussian. Because S is fitted to these same rows the reference
    is a scaled Beta, not chi-squared; the difference matters in the tail, which is what we read."""
    B = np.column_stack([cond, np.ones(len(cond))])
    X = X - B @ np.linalg.lstsq(B, X, rcond=None)[0]
    n, p = X.shape
    Xc = X - X.mean(0)
    d2 = np.einsum('ij,jk,ik->i', Xc, np.linalg.inv(np.cov(Xc, rowvar=False)), Xc)
    q = (np.arange(1, n + 1) - 0.5) / n
    ref = ((n - 1)**2 / n) * stats.beta.ppf(q, p/2, (n - p - 1)/2)
    return np.sort(d2), ref

In [ ]:
# Figure 3 -- Mahalanobis QQ per block. On the red line = jointly Gaussian at this dimension.
from matplotlib.ticker import NullFormatter

fig, axes = plt.subplots(1, len(BLOCKS), figsize=(2.6*len(BLOCKS), 2.9))
for ax, (name, (X, cond)) in zip(axes, BLOCKS.items()):
    d2, ref = mahalanobis_qq(X, cond)
    i = np.unique(np.linspace(0, len(d2) - 1, 800).astype(int))
    k = int(0.99 * len(d2))                      # tail ratio: observed / expected at p99
    lo, hi = min(ref[0], d2[0]), max(ref[-1], d2[-1])
    ax.plot([lo, hi], [lo, hi], color=RED, lw=1.1, zorder=1)
    ax.plot(ref[i], d2[i], 'o', ms=2, color=BLUE, alpha=.7, zorder=2)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(lo*.8, hi*1.25); ax.set_ylim(lo*.8, hi*1.25)   # equal limits -> 45° diagonal
    ax.xaxis.set_minor_formatter(NullFormatter())               # decade labels only; the narrow
    ax.yaxis.set_minor_formatter(NullFormatter())               # panels crowd otherwise
    ax.set_title(f'{name}\nd={X.shape[1]}   tail ratio {d2[k]/ref[k]:.1f}x', pad=4)
    ax.set_xlabel('expected d²  (Gaussian)')
axes[0].set_ylabel('observed d²')
fig.suptitle('Joint Gaussianity — Mahalanobis QQ, log–log', y=1.04, fontsize=10)
fig.tight_layout()
plt.show()

### Reading this

* **On the line** — no departure from joint Gaussianity detectable at this dimension.
* **Curving above the line at the right** — heavier joint tails than Gaussian: rows where many
  channels are extreme *together*. Those rows dominate `S`, and `S` is the estimator's only input.
* **`tail ratio`** — observed ÷ expected `d²` at the 99th percentile. `1.0` = Gaussian. Compare every
  panel against the `gaussian control`, which shows the sampling noise at this `n` and `d`.

If `adjacency X+T` departs much more than `adjacency X` or `adjacency T` alone, the failure is in
the *cross*-block structure — the part the MI actually depends on — rather than in either block
separately.

Same caveat as Part 1: this pools all timesteps, and a scale mixture over `t` is heavy-tailed even
when every slice is Gaussian. So a failure here does not yet distinguish claim 2 from claim 3.